[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haohanchen/POLI3148_2026Spring_TextAnalysis/blob/main/lecture/session_1_text_as_data/session_1_text_as_data.ipynb)

# Session 1: Text as Data -- Concepts and First Analysis

**POLI3148 Data Science in Politics and Public Administration**
The University of Hong Kong | Dr. Chen Haohan

---

In this session, we explore the building blocks of **text analysis** -- one of the most powerful tools in modern social science. We will work with a real dataset of press conference transcripts from China's Ministry of Foreign Affairs (MoFA), and learn how to:

- Understand the structure of a text corpus
- Break text into meaningful units (tokenization, lemmatization)
- Identify named entities (people, places, organizations) automatically
- Process multiple documents programmatically

By the end of this session, you will have the foundational vocabulary and hands-on skills to start analyzing text data in your own research.

## Setup

First, let's install and import the tools we need. **spaCy** is an industrial-strength natural language processing (NLP) library that we will use for tokenization and named entity recognition.

In [ ]:
!pip install -q spacy openpyxl
!python -m spacy download en_core_web_sm -q

In [ ]:
import pandas as pd
import spacy
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_colwidth", 30)

---
## Part 1: Meet the Data -- China's MoFA Press Conferences

The **Chinese Ministry of Foreign Affairs (MoFA) Press Conference Corpus** contains **35,346 question-and-answer dyads** from regular press conferences held between 2002 and 2025. Each row is one exchange: a journalist asks a question, and a MoFA spokesperson answers.

The dataset was compiled by academic researchers and published on [Harvard Dataverse](https://dataverse.harvard.edu/). It includes the raw text of questions and answers, pre-computed linguistic features (lemmatized text, named entities, sentiment scores), and metadata (date, spokesperson, news agency).

This is a rich resource for studying Chinese foreign policy, diplomatic communication, and media relations.

In [ ]:
# Load the dataset
df = pd.read_excel("https://github.com/haohanchen/POLI3148_2026Spring_TextAnalysis/raw/main/data/CMFA_PressCon_v6.xlsx")
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
display(df.head())

In [ ]:
# Display core columns: ID, spokesperson, date, question, answer, entities, sentiment
core_cols = ["id", "spokesperson", "date", "question", "answer",
             "q_loc", "q_per", "q_org", "q_sentiment", "a_sentiment"]
display(df[core_cols].head(10))

In [ ]:
# Look at a single Q&A exchange in full
row = df.iloc[100]
print(f"Date: {row['date']}")
print(f"Spokesperson: {row['spokesperson']}")
print(f"Asked by: {row['who_asked']}")
print(f"\n--- QUESTION ---\n{row['question']}")
print(f"\n--- ANSWER ---\n{row['answer']}")

In [ ]:
# Spokesperson distribution
fig, ax = plt.subplots(figsize=(10, 5))
df["spokesperson"].value_counts().head(10).plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("Number of Q&A exchanges")
ax.set_title("Top 10 MoFA Spokespersons by Number of Press Conference Q&As")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Number of Q&A exchanges per year
yearly = df.groupby("year").size()

fig, ax = plt.subplots(figsize=(10, 4))
yearly.plot(kind="line", marker="o", ax=ax, color="steelblue")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Q&A exchanges")
ax.set_title("MoFA Press Conference Q&As Over Time")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 2: Key Concepts -- Units of Text Analysis

When we analyze text computationally, we think in terms of a hierarchy of units:

| Unit | Definition | Example |
|------|-----------|---------|
| **Corpus** | The whole collection | All 35,346 Q&A pairs |
| **Document** | A single text unit | One question or one answer |
| **Paragraph / Sentence** | Sub-document units | A sentence within an answer |
| **Token** | The smallest unit | A word, punctuation mark, or number |

**Corpus → Documents → Sentences → Tokens**

Most analysis operates at the **document** or **token** level. But a sentence or paragraph can also be a unit of analysis (e.g., sentence-level sentiment, sentence embeddings).

We can ask questions at every level:

- **Corpus level:** What topics dominate MoFA press conferences? What are the most frequently mentioned countries?
- **Document level:** What topic does this question discuss? Is this question aggressive? What countries and people are mentioned?
- **Across documents:** How do different spokespeople differ? How has diplomatic tone changed over time?
- **Token level:** What are the most common words? How often does "sovereignty" appear?

The tools we learn today -- tokenization, lemmatization, and named entity recognition -- help us move between these levels systematically.

---
## Part 3: Tokenization

### What is tokenization?

**Tokenization** is the process of breaking a text into individual units called **tokens**. A token is typically a word or punctuation mark.

> "China opposes the decision." -> ["China", "opposes", "the", "decision", "."]

### What is lemmatization?

**Lemmatization** reduces words to their base or dictionary form (the **lemma**). This helps us group together different forms of the same word.

> "opposes" -> "oppose" | "countries" -> "country" | "was" -> "be"

### What is POS tagging?

**Part-of-speech (POS) tagging** labels each token with its grammatical role: noun, verb, adjective, adverb, etc.

### What are stopwords?

**Stopwords** are extremely common words (like "the", "is", "a", "of") that usually carry little meaning on their own. Many text analysis pipelines remove them to focus on content words.

Let's see all of this in action using **spaCy**.

In [ ]:
# Load the spaCy English model
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Pick a question from the dataset and process it with spaCy
sample_idx = 4
sample_question = df.iloc[sample_idx]["question"]
print(f"Row {sample_idx} question:\n{sample_question}\n")

doc = nlp(sample_question)

In [ ]:
# Display tokens as a table
token_data = []
for token in doc:
    token_data.append({
        "Text": token.text,
        "Lemma": token.lemma_,
        "POS": token.pos_,
        "Is Stopword": token.is_stop
    })

token_df = pd.DataFrame(token_data)
display(token_df)

In [ ]:
# Show words where lemmatization made a change
changed = token_df[token_df["Text"].str.lower() != token_df["Lemma"].str.lower()]
print("Tokens where lemmatization changed the word:")
display(changed[["Text", "Lemma", "POS"]])

In [ ]:
# Compare with the dataset's pre-computed lemmatized version
print("--- spaCy lemmatized (our output) ---")
spacy_lemmas = " ".join([token.lemma_ for token in doc if not token.is_punct])
print(spacy_lemmas[:500])

print("\n--- Dataset's pre-computed lemmatized version ---")
print(str(df.iloc[sample_idx]["question_lem"])[:500])

---
## Part 4: Named Entity Recognition (NER)

**Named Entity Recognition (NER)** is the task of automatically identifying and classifying named entities in text -- such as people, organizations, locations, dates, and more.

This is one of the most practically useful NLP tasks. For political science, NER lets us automatically extract **which countries, leaders, or organizations** are being discussed in the documents.

Let's see how spaCy performs NER on our MoFA questions.

In [ ]:
# Run NER on our sample question
print(f"Question: {sample_question}\n")
print("Named entities found by spaCy:")
for ent in doc.ents:
    print(f"  {ent.text:30s} -> {ent.label_:10s} ({spacy.explain(ent.label_)})")

In [ ]:
# Try NER on a few more entity-rich questions
for idx in [50, 200, 500]:
    q = df.iloc[idx]["question"]
    doc_temp = nlp(q)
    print(f"--- Row {idx} ---")
    print(f"Question: {q[:200]}...")
    for ent in doc_temp.ents:
        print(f"  {ent.text:30s} -> {ent.label_}")
    print()

In [ ]:
# Compare spaCy NER with the dataset's pre-labeled entities
for idx in [4, 50, 200]:
    q = df.iloc[idx]["question"]
    doc_temp = nlp(q)

    # spaCy entities grouped by type
    spacy_locs = [ent.text for ent in doc_temp.ents if ent.label_ == "GPE"]
    spacy_pers = [ent.text for ent in doc_temp.ents if ent.label_ == "PERSON"]
    spacy_orgs = [ent.text for ent in doc_temp.ents if ent.label_ == "ORG"]

    # Dataset entities
    ds_locs = str(df.iloc[idx]["q_loc"])
    ds_pers = str(df.iloc[idx]["q_per"])
    ds_orgs = str(df.iloc[idx]["q_org"])

    print(f"--- Row {idx} ---")
    print(f"  Question: {q[:150]}...")
    print(f"  Locations  | spaCy: {spacy_locs}  |  Dataset: {ds_locs}")
    print(f"  Persons    | spaCy: {spacy_pers}  |  Dataset: {ds_pers}")
    print(f"  Orgs       | spaCy: {spacy_orgs}  |  Dataset: {ds_orgs}")
    print()

### Why do NER tools disagree?

You will notice that spaCy's output does not always match the dataset's pre-labeled entities. The dataset authors used a different NER tool called **Flair**. Disagreements between NER tools are common and arise because:

1. **Different training data.** Each model learns from different annotated corpora, so they develop different strengths and blind spots.
2. **Different entity categories.** spaCy distinguishes GPE (countries, cities) from LOC (non-political locations like mountains), while other tools may merge them.
3. **Ambiguity in text.** Is "Washington" a person, a city, or a state? Context helps, but models interpret it differently.
4. **Model size and architecture.** We are using spaCy's small model (`en_core_web_sm`). Larger models tend to be more accurate.

This is an important lesson: **NLP tools are not perfect.** Results always require critical evaluation.

---
## Part 5: Processing Multiple Documents

So far we have processed one document at a time. In practice, we need to process many documents and collect the results systematically. Let's process a batch of questions and extract their named entities into a structured DataFrame.

In [ ]:
# Process 20 questions and extract NER results
results = []

for i in range(20):
    question = df.iloc[i]["question"]
    doc_temp = nlp(str(question))

    locations = [ent.text for ent in doc_temp.ents if ent.label_ in ("GPE", "LOC")]
    persons = [ent.text for ent in doc_temp.ents if ent.label_ == "PERSON"]
    orgs = [ent.text for ent in doc_temp.ents if ent.label_ == "ORG"]

    results.append({
        "row": i,
        "year": df.iloc[i]["year"],
        "question_preview": str(question)[:80] + "...",
        "locations": "; ".join(locations) if locations else "-",
        "persons": "; ".join(persons) if persons else "-",
        "organizations": "; ".join(orgs) if orgs else "-",
    })

results_df = pd.DataFrame(results)
display(results_df)

**Scaling up.** The loop above works fine for 20 documents. For the full dataset of 35,000+ documents, spaCy provides `nlp.pipe()`, which processes texts in batches and is significantly faster:

```python
# Example (do not run -- takes several minutes on the full dataset)
docs = nlp.pipe(df["question"].astype(str).tolist(), batch_size=50)
for doc in docs:
    # extract entities, tokens, etc.
    pass
```

We will explore batch processing further in later sessions.

---
## Your Turn

### Exercise 1: Tokenize MoFA answers

Pick **5 rows** from the `answer` column. Tokenize each one with spaCy. Then:

1. Collect all lemmas (excluding stopwords and punctuation) across all 5 answers.
2. Count how often each lemma appears.
3. What are the **10 most common lemmas**?

*Hint: Use a list or `collections.Counter` to accumulate lemma counts.*

In [ ]:
# Exercise 1: Your code here


### Exercise 2: NER on MoFA answers

Using the **same 5 rows**, run spaCy NER on each answer. Then:

1. For each row, extract the locations, persons, and organizations that spaCy finds.
2. Compare your results with the dataset's pre-labeled columns: `a_loc`, `a_per`, `a_org`.
3. Where do they agree? Where do they disagree? Why might that be?

In [ ]:
# Exercise 2: Your code here


### Exercise 3: Vibe coding challenge

Copy the prompt below into an AI coding assistant (e.g., ChatGPT, Claude, GitHub Copilot) and use the generated code to extend this notebook:

> "Modify this notebook to process all questions from a single year (e.g., 2022). Extract all named locations using spaCy and count how often each country/location appears. Which countries were discussed most in MoFA press conferences that year? Show the top 20 as a bar chart."

After pasting the AI-generated code below, **review it carefully** before running. Does the code look correct? Does it handle edge cases (e.g., missing values)?

In [ ]:
# Exercise 3: Paste and run AI-generated code here
